# HPA Normal-Tissue Safety Filtering

**v0.1 educational notebook**

## Goal
Add normal-tissue protein evidence and critical-organ risk.

**Data mode:** Synthetic HPA-style executable example. Real HPA files should be downloaded from the original provider.

## Canonical guide

### Learning objectives
- Represent normal-tissue protein burden and critical-organ exposure risk.
- Preserve the distinction between “Not detected” and missing/unknown.
- Create protein-safety features compatible with Notebook 04.

### Data mode
Executable core: synthetic HPA-style IHC table. Real-data extension: current HPA downloadable normal-tissue IHC data.

### Inputs
`outputs/tcga_gtex_expression_features.csv` (used only to keep the same gene universe in v0.1).

### Canonical outputs
`outputs/hpa_normal_tissue_features.csv`

### Scientific limitation
Normal-tissue IHC expression is not equivalent to extracellular accessibility, internalization, systemic exposure, or clinical toxicity.

### Next step
Notebook 04 integrates expression, protein safety, and surface annotation.


In [1]:
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'
OUTPUTS = ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)
print('ROOT:', ROOT.resolve())

ROOT: /mnt/data/audit_adc/adc-scientific-ai


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
rng=np.random.default_rng(43)
expr=pd.read_csv(OUTPUTS/'tcga_gtex_expression_features.csv',index_col=0)
genes=expr.index.tolist()
tissues=['Heart muscle','Lung','Liver','Kidney','Bone marrow','Colon','Skin','Breast']
levels=['Not detected','Low','Medium','High']
level_map={'Not detected':0,'Low':1,'Medium':2,'High':3}
rows=[]
for g in genes:
    probs=[0.45,0.35,0.15,0.05]
    if g in ['EPCAM','GENE_F','GENE_K']: probs=[0.10,0.25,0.40,0.25]
    for t in tissues:
        level=rng.choice(levels,p=probs)
        rows.append((g,t,'Representative cell type',level,'Approved'))
hpa=pd.DataFrame(rows,columns=['gene_symbol','tissue','cell_type','level','reliability'])
hpa['level_numeric']=hpa.level.map(level_map)
hpa.head()

,gene_symbol,tissue,cell_type,level,reliability,level_numeric
0,ERBB2,Heart muscle,Representative cell type,Low,Approved,1
1,ERBB2,Lung,Representative cell type,Not detected,Approved,0
2,ERBB2,Liver,Representative cell type,Not detected,Approved,0
3,ERBB2,Kidney,Representative cell type,Medium,Approved,2
4,ERBB2,Bone marrow,Representative cell type,Low,Approved,1


In [3]:
critical={'Heart muscle','Lung','Liver','Kidney','Bone marrow'}
def build_hpa_features(df):
    out=pd.DataFrame(index=sorted(df.gene_symbol.unique()))
    out.index.name='gene_symbol'
    g=df.groupby('gene_symbol')
    out['hpa_normal_max']=g.level_numeric.max()
    out['n_normal_tissues_positive']=g.apply(lambda x:x.loc[x.level_numeric>0,'tissue'].nunique(), include_groups=False)
    out['normal_tissue_fraction_positive']=out.n_normal_tissues_positive/df.tissue.nunique()
    out['n_normal_celltypes_positive']=g.apply(lambda x:x.loc[x.level_numeric>0,'cell_type'].nunique(), include_groups=False)
    crit=df[df.tissue.isin(critical)]
    out['critical_organ_max']=crit.groupby('gene_symbol').level_numeric.max().reindex(out.index)
    out['n_critical_tissues_positive']=crit.groupby('gene_symbol').apply(lambda x:x.loc[x.level_numeric>0,'tissue'].nunique(), include_groups=False).reindex(out.index)
    out['normal_protein_risk']=(0.4*(out.hpa_normal_max/3)+0.4*(out.critical_organ_max/3)+0.2*out.normal_tissue_fraction_positive)
    out['hpa_missing']=False
    return out
hpa_features=build_hpa_features(hpa)
hpa_features.head()

,hpa_normal_max,n_normal_tissues_positive,normal_tissue_fraction_positive,n_normal_celltypes_positive,critical_organ_max,n_critical_tissues_positive,normal_protein_risk,hpa_missing
gene_symbol,,,,,,,,
EPCAM,2,7,0.875,1,2,4,0.708333,False
ERBB2,2,4,0.500,1,2,3,0.633333,False
FOLR1,3,3,0.375,1,3,2,0.875000,False
GENE_A,2,4,0.500,1,1,2,0.500000,False
GENE_B,2,2,0.250,1,1,1,0.450000,False


In [4]:
path=OUTPUTS/'hpa_normal_tissue_features.csv'
hpa_features.to_csv(path)
print('saved:',path)

saved: /mnt/data/audit_adc/adc-scientific-ai/outputs/hpa_normal_tissue_features.csv


## Scientific caution
`Not detected` is an observed category; missing/unknown data are different. Normal-tissue expression is not the same as ADC accessibility or clinical toxicity. Exposure, internalization, linker stability, payload properties, and assay context all matter.